In [1]:
import pandas as pd
import numpy as np
import pickle

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
df = pd.read_csv("final_movies.csv")

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   movie_id         500 non-null    int64
 1   title            500 non-null    str  
 2   genres           500 non-null    str  
 3   overview         500 non-null    str  
 4   overview_length  500 non-null    int64
dtypes: int64(2), str(3)
memory usage: 66.1 KB


In [4]:
movies = df[['movie_id', 'title', 'genres', 'overview']]

In [5]:
movies.isnull().sum()

movie_id    0
title       0
genres      0
overview    0
dtype: int64

In [6]:
movies['genres'] = movies['genres'].str.lower()
movies['overview'] = movies['overview'].str.lower()

In [7]:
movies['genres'] = movies['genres'].str.strip()
movies['overview'] = movies['overview'].str.strip()

In [8]:
movies['tags'] = movies['genres'] + " " + movies['overview']

In [9]:
movies[['title','tags']].head()

,title,tags
0,Shadow Journey,drama an unexpected discovery changes the live...
1,Shadow Empire,"drama a heartfelt journey explores friendship,..."
2,Shadow Legacy,romance|drama an unexpected discovery changes ...
3,Shadow Promise,action|sci-fi an unexpected discovery changes ...
4,Shadow Storm,horror|mystery an unexpected discovery changes...


In [10]:
movies = movies[['movie_id','title','tags']]

In [11]:
movies.drop_duplicates(subset='title', inplace=True)
movies.reset_index(drop=True, inplace=True)

In [12]:
movies.head()

,movie_id,title,tags
0,1,Shadow Journey,drama an unexpected discovery changes the live...
1,2,Shadow Empire,"drama a heartfelt journey explores friendship,..."
2,3,Shadow Legacy,romance|drama an unexpected discovery changes ...
3,4,Shadow Promise,action|sci-fi an unexpected discovery changes ...
4,5,Shadow Storm,horror|mystery an unexpected discovery changes...


In [13]:
movies.shape

(500, 3)

In [14]:
cv = CountVectorizer(
    max_features=5000,
    stop_words='english'
)

In [15]:
vectors = cv.fit_transform(movies['tags']).toarray()

In [16]:
vectors.shape

(500, 55)

In [17]:
similarity = cosine_similarity(vectors)

In [18]:
similarity.shape

(500, 500)

In [19]:
def recommend(movie_name):

    movie_name = movie_name.lower()

    movies['title_lower'] = movies['title'].str.lower()

    if movie_name not in movies['title_lower'].values:
        print("Movie not found!")
        return

    index = movies[movies['title_lower'] == movie_name].index[0]

    distances = similarity[index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    print("Recommended Movies:\n")

    for i in movie_list:
        print(movies.iloc[i[0]].title)

In [20]:
recommend("Avatar")

Movie not found!


In [21]:
pickle.dump(movies, open("movies.pkl", "wb"))

In [22]:
pickle.dump(similarity, open("similarity.pkl", "wb"))

In [23]:
movies = pickle.load(open("movies.pkl", "rb"))
similarity = pickle.load(open("similarity.pkl", "rb"))

In [24]:
recommend("Avatar")

Movie not found!
